### code to translate German to English and vice versa with deepL

In [2]:
import pandas as pd
from pprint import pprint
import os
import ast
import deepl
import json
import requests

In [3]:
# get connection to deepl api
deepl_key = open("/Users/sofiaromero/Documents/DeepL.txt").read()
deepl_client = deepl.DeepLClient(deepl_key)

In [4]:
# api functions 

def generate_glossaries(dict_path_l, s_lang, t_lang, auth_key):
    url = "https://api-free.deepl.com/v3/glossaries"
    header= {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
        "Content-Type": "application/json"
    }    
    glossaries = []
    for i,d in enumerate(dict_path_l):
        payload = {
            "name": f"glossary{i}",
            "dictionaries": [{
                # TODO: get languages from file name
                "source_lang": s_lang,
                "target_lang": t_lang,
                "entries": open(d,"r").read(),
                "entries_format": "csv"}
            ]}
        resp = requests.post(url, headers=header, json=payload)
        print(resp.status_code, resp.json())
        glossaries.append(resp.json())
    return glossaries


def get_glossary_ids(auth_key):
    url = "https://api-free.deepl.com/v3/glossaries"
    headers = {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
    }
    resp = requests.get(url, headers=headers, timeout=30)
    print(resp.status_code)
    return [x.get("glossary_id") for x in resp.json()["glossaries"]]


def transalate_to(text_l, s_lang, t_lang, glossary_id):
    res = []
    for text in text_l:
        if glossary_id:
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang, glossary=glossary_id).text)
        else: 
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang).text)
    return res


In [5]:
#generate_glossaries(["config_files/ger_to_eng.csv"], "DE", "EN", deepl_key)
# get all current glossaries
glossary_ids = get_glossary_ids(deepl_key)

200


In [6]:
# translate ki terms as test
'''
test_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
print(test_terms)
translations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])
print(translations)
'''

'\ntest_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())\nprint(test_terms)\ntranslations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])\nprint(translations)\n'

In [7]:
# write text (from file) and its transaltion to df and save to csv and json file
def tranlate_to_df(txts_json, file_to):
    # load json as dict
    with open(txts_json, "r") as f:
        txts = json.load(f)

    res_df = pd.DataFrame([(key, var) for (key, L) in txts.items() for var in L], #[(key, var) for (key, var) in txts.items()],
                 columns=['id', 'text'])
    #fill df with traslations
    res_df['translation'] = transalate_to(res_df['text'].tolist(), "DE", "EN-GB", glossary_ids[0])
    #save to file 
    res_df.to_csv(f"{file_to}.csv", index=False)
    res_df.to_json(f"{file_to}.json", orient="records")
    return res_df

#test translation and save to df
tranlate_to_df("generated_data/KI_sentences_mock.json", "generated_data/anthroscore_KI_terms_translated")

,id,text,translation
0,1,Apfel,Apple
1,1,Rote Frucht,Red fruit
2,2,Auto,Car
3,2,Fahrzeug mit vier Rädern,Vehicle with four wheels
4,3,Buch,Book
5,3,Sammlung beschriebener Seiten,Collection of written pages
6,4,Computer in großer Höhe,Computers at great heights
7,4,Elektronisches Gerät zur Datenverarbeitung,Electronic device for data processing
8,5,Ozean,Ocean
9,5,Großes Salzwassergewässer,Large saltwater body of water


### Translate KI_sentences to English

In [8]:
tranlate_to_df("generated_data/KI_sentences.json", "generated_data/KI_sentences_translated")

,id,text,translation
0,_5774798.txt,palm recyclingpapier a beig,palm recycled paper a beig
1,_1825405.txt,pikanterweise hält man nämlich mit dem erzriva...,"piquantly, the company holds a joint venture w..."
2,_1825405.txt,man will jetzt die kontrolle über gpt allein a...,you now want to exercise sole control over gpt
3,_5569995.txt,die welt und ich 2000 ki,the world and me 2000 ki
4,_5571102.txt,palm recyclingpapier a beig,palm recycled paper a beig
...,...,...,...
1975,_5911925.txt,palm recyclingpapier a beig,palm recycled paper a beig
1976,_1736092.txt,martina palm studentin hamburg,martina palm student hamburg
1977,_0403353.txt,ungeneralsekretär ban ki moon nannte tunesien ...,secretary-general ban ki moon called tunisia a...
1978,_1398983.txt,cdu bremenstadt was würde wohl passieren wenn ...,cdu bremenstadt what would happen if the liber...
